In [1]:
from bs4 import BeautifulSoup
from selenium import webdriver      
from selenium.webdriver.common.by import By
import time 
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import random
import requests
from IPython.display import display
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service

In [2]:
'''url = 'https://www.dges.gov.pt/guias/indcurso.asp'
browser = webdriver.Chrome(options=Options())  # Initialize a Chrome browser instance with options
browser.get(url)  # Open the URL in the browser
time.sleep(1)  '''

"url = 'https://www.dges.gov.pt/guias/indcurso.asp'\nbrowser = webdriver.Chrome(options=Options())  # Initialize a Chrome browser instance with options\nbrowser.get(url)  # Open the URL in the browser\ntime.sleep(1)  "

In [3]:
'''def scrape_current_letter(browser, course_institution_data):
    """Scrape all courses & institutions from the current letter page."""
    WebDriverWait(browser, 20).until(
        EC.presence_of_all_elements_located((By.CLASS_NAME, "box10"))
    )

    soup = BeautifulSoup(browser.page_source, "html.parser")
    all_blocks = soup.find_all(["div"], class_=["box10", "lin-curso"])
    current_course = None

    for block in all_blocks:
        classes = block.get("class", [])

        # --- Course block ---
        if "box10" in classes:
            name_tag = block.find("div", class_="lin-area-c2")
            if name_tag:
                current_course = name_tag.text.strip()
                print(f"\n📘 Course: {current_course}")

        # --- Institution block ---
        elif "lin-curso" in classes and current_course:
            link_tag = block.find("a")
            if not link_tag:
                continue

            institution = link_tag.text.strip()
            href = link_tag.get("href")

            if any(kw in institution for kw in ["Universidade", "Instituto", "Escola", "Politécnico"]):
                print(f"🏫 Institution: {institution}")

                try:
                    # Click the institution
                    inst_element = WebDriverWait(browser, 10).until(
                        EC.element_to_be_clickable((By.XPATH, f"//a[@href='{href}']"))
                    )
                    browser.execute_script("arguments[0].scrollIntoView(true);", inst_element)
                    time.sleep(0.3)
                    inst_element.click()

                    # Wait for detail page
                    WebDriverWait(browser, 15).until(
                        EC.presence_of_all_elements_located((By.CLASS_NAME, "inside2"))
                    )
                    time.sleep(1)

                    # Parse the detail page
                    detail_soup = BeautifulSoup(browser.page_source, "html.parser")

                    # --- Extract Google Maps link ---
                    google_map = ""
                    inside_block = detail_soup.find("div", class_="inside2")
                    if inside_block:
                        map_link = inside_block.find("a", href=True, string=lambda t: t and "Mapa" in t)
                        if not map_link:
                            map_span = inside_block.find("span", class_="vislink", string=lambda t: "Mapa" in t)
                            if map_span and map_span.parent.name == "a":
                                map_link = map_span.parent
                        if map_link:
                            google_map = map_link["href"].strip()

                    print(f"🗺️ Google Maps link: {google_map if google_map else 'Not found'}")

                except Exception as e:
                    print(f"⚠️ Error scraping {institution}: {e}")
                    google_map = ""

                # Go back to the list page
                browser.back()
                time.sleep(1)
                WebDriverWait(browser, 20).until(
                    EC.presence_of_all_elements_located((By.CLASS_NAME, "box10"))
                )

                # Store
                course_institution_data.append((current_course, institution, href, google_map))

    return course_institution_data


# --- Step 1: Scrape letter A (already selected) ---
print("\n🔤 Scraping letter: A (default)")
course_institution_data = []
course_institution_data = scrape_current_letter(browser, course_institution_data)

# --- Step 2: Click and scrape all other letters ---
letters = browser.find_elements(By.CSS_SELECTOR, "div.noprint a")
letter_links = [(a.text.strip(), a.get_attribute("href")) for a in letters if a.text.strip()]

for letter, link in letter_links:
    print(f"\n🔤 Scraping letter: {letter}")
    browser.get(link)
    time.sleep(1.5)
    course_institution_data = scrape_current_letter(browser, course_institution_data)

# --- Save results ---
df = pd.DataFrame(course_institution_data, columns=["Course", "Institution", "Link", "GoogleMaps"])'''

'def scrape_current_letter(browser, course_institution_data):\n    """Scrape all courses & institutions from the current letter page."""\n    WebDriverWait(browser, 20).until(\n        EC.presence_of_all_elements_located((By.CLASS_NAME, "box10"))\n    )\n\n    soup = BeautifulSoup(browser.page_source, "html.parser")\n    all_blocks = soup.find_all(["div"], class_=["box10", "lin-curso"])\n    current_course = None\n\n    for block in all_blocks:\n        classes = block.get("class", [])\n\n        # --- Course block ---\n        if "box10" in classes:\n            name_tag = block.find("div", class_="lin-area-c2")\n            if name_tag:\n                current_course = name_tag.text.strip()\n                print(f"\n📘 Course: {current_course}")\n\n        # --- Institution block ---\n        elif "lin-curso" in classes and current_course:\n            link_tag = block.find("a")\n            if not link_tag:\n                continue\n\n            institution = link_tag.text.strip(

In [4]:
'''
# ---------------------------------------------------
# START BROWSER
# ---------------------------------------------------
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
wait = WebDriverWait(driver, 15)

BASE_URL = "https://eduportugal.eu/cursos-estudo/mestrado/"
driver.get(BASE_URL)

results = []

# ---------------------------------------------------
# FIND TOTAL NUMBER OF PAGES
# ---------------------------------------------------
wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, ".wp-pagenavi")))
last_page = driver.find_element(By.CSS_SELECTOR, ".wp-pagenavi .last").get_attribute("href")
total_pages = int(last_page.rstrip("/").split("/")[-1])

print(f"🔎 Total pages detected: {total_pages}\n")


# ---------------------------------------------------
# LOOP THROUGH ALL PAGES
# ---------------------------------------------------
for page in range(1, total_pages + 1):
    if page == 1:
        url = BASE_URL
    else:
        url = f"{BASE_URL}page/{page}/"

    print(f"\n==============================")
    print(f"📄 Scraping page {page} of {total_pages}")
    print(f"==============================\n")

    driver.get(url)
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "tbody tr")))

    # Get course links on this page
    rows = driver.find_elements(By.CSS_SELECTOR, "tbody tr")
    course_urls = []

    for row in rows:
        link = row.find_element(By.CSS_SELECTOR, "td:nth-child(1) a").get_attribute("href")
        course_urls.append(link)

    print(f"   → Found {len(course_urls)} courses on this page.\n")

    # ---------------------------------------------------
    # VISIT EACH COURSE
    # ---------------------------------------------------
    for i, course_url in enumerate(course_urls, start=1):
        print(f"   [{i}/{len(course_urls)}] Visiting: {course_url}")

        driver.get(course_url)
        time.sleep(2)

        # Extract master name
        try:
            master_name = driver.find_element(By.CSS_SELECTOR, "div.col-md-8 h1").text.strip()
        except:
            master_name = "N/A"

        # Extract university
        try:
            university = driver.find_element(By.CSS_SELECTOR, "h6.media-heading a").text.strip()
        except:
            university = "N/A"

        # Extract about the course
        try:
            about_container = driver.find_element(By.CSS_SELECTOR, "div.content-dropcap")
            about = about_container.text.strip()
        except:
            about = ""

        results.append({
            "url": course_url,
            "master": master_name,
            "university": university,
            "about": about
        })

        print(f"        ✓ Master: {master_name}")
        print(f"        ✓ University: {university}\n")

driver.quit()'''

'\n# ---------------------------------------------------\n# START BROWSER\n# ---------------------------------------------------\ndriver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))\nwait = WebDriverWait(driver, 15)\n\nBASE_URL = "https://eduportugal.eu/cursos-estudo/mestrado/"\ndriver.get(BASE_URL)\n\nresults = []\n\n# ---------------------------------------------------\n# FIND TOTAL NUMBER OF PAGES\n# ---------------------------------------------------\nwait.until(EC.presence_of_element_located((By.CSS_SELECTOR, ".wp-pagenavi")))\nlast_page = driver.find_element(By.CSS_SELECTOR, ".wp-pagenavi .last").get_attribute("href")\ntotal_pages = int(last_page.rstrip("/").split("/")[-1])\n\nprint(f"🔎 Total pages detected: {total_pages}\n")\n\n\n# ---------------------------------------------------\n# LOOP THROUGH ALL PAGES\n# ---------------------------------------------------\nfor page in range(1, total_pages + 1):\n    if page == 1:\n        url = BASE_URL\n    else:\

In [5]:
# --------------------- Chrome setup ---------------------
options = Options()
options.add_experimental_option("debuggerAddress", "127.0.0.1:9222")
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 20)

# --------------------- Lists ---------------------
names = []
universities = []
locations = []
durations = []
tuitions = []
abouts = []

seen = set()  # avoid duplicates

# --------------------- Loop over numbered pages ---------------------
page = 17
while True:

    url = f"https://www.mastersportal.com/search/master/portugal?page={page}"
    print(f"\n=== Loading page {page} ===")
    driver.get(url)

    # check if page has any cards
    try:
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "a.SearchStudyCard")))
    except:
        print("No more results. End of pagination.")
        break

    # scroll to load all cards
    previous_len = 0
    same_count_rounds = 0
    while True:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(1.5)

        cards = driver.find_elements(By.CSS_SELECTOR, "a.SearchStudyCard")
        new_len = len(cards)

        if new_len == previous_len:
            same_count_rounds += 1
        else:
            same_count_rounds = 0

        if same_count_rounds >= 3:
            break

        previous_len = new_len

    cards = driver.find_elements(By.CSS_SELECTOR, "a.SearchStudyCard")
    total_cards = len(cards)
    print(f"Found {total_cards} cards.")

    if total_cards == 0:
        print("No cards on this page → stopping.")
        break

    # -------- extract each card --------
    for i in range(total_cards):

        # re-find fresh card
        cards = driver.find_elements(By.CSS_SELECTOR, "a.SearchStudyCard")
        card = cards[i]

        try:
            name = card.find_element(By.CSS_SELECTOR, "h2.StudyName").text
        except:
            name = ""

        try:
            uni = card.find_element(By.CSS_SELECTOR, "strong.OrganisationName").text
        except:
            uni = ""

        key = (name, uni)
        if key in seen or name == "":
            continue
        seen.add(key)

        print(f"\nProcessing: {name} | {uni}")

        try:
            loc = card.find_element(By.CSS_SELECTOR, "strong.OrganisationLocation").text
        except:
            loc = ""

        try:
            duration = card.find_element(By.CSS_SELECTOR, ".DurationValue").text
        except:
            duration = ""

        try:
            tuition = card.find_element(By.CSS_SELECTOR, ".TuitionValue").text
        except:
            tuition = ""

        # --- go to program page ---
        try:
            program_link = card.get_attribute("href")
            driver.get(program_link)

            wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "section#StudySummary")))
            time.sleep(1)

            about = driver.find_element(By.CSS_SELECTOR, "section#StudySummary p").text

            print("About extracted.")
            driver.back()
            wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "a.SearchStudyCard")))
            time.sleep(1)

        except Exception as e:
            print("Failed About:", e)
            about = ""

        names.append(name)
        universities.append(uni)
        locations.append(loc)
        durations.append(duration)
        tuitions.append(tuition)
        abouts.append(about)

    page += 1  # go to next page number

print("\n=== SCRAPING COMPLETE ===")


=== Loading page 17 ===
Found 20 cards.

Processing: Management Information Systems | University of Lisbon
About extracted.

Processing: Chemical and Biological Engineering | Universidade Nova de Lisboa
About extracted.

Processing: International Hospitality Management | SHG Universities
About extracted.

Processing: International Master of Business Administration | Porto Business School
About extracted.

Processing: Building Rehabilitation | Universidade Nova de Lisboa
About extracted.

Processing: Civil Engineering | Universidade Nova de Lisboa
About extracted.

Processing: Physical Activity and Sports | University of Madeira
About extracted.

Processing: International Business | Porto Business School
About extracted.

Processing: Cultural Management | University of Madeira
About extracted.

Processing: Global Online Master of Business Administration | Porto Business School
About extracted.

Processing: Education Sciences - Educational Administration | University of Madeira
About ex

KeyboardInterrupt: 

In [6]:
# Create DataFrame
df = pd.DataFrame({
    "Master Name": names,
    "University": universities,
    "Location": locations,
    "Duration": durations,
    "Tuition Fee": tuitions,
    "About": abouts
})
df
#df.to_csv("first_two_pages.csv", index=False, encoding="utf-8")

,Master Name,University,Location,Duration,Tuition Fee,About
0,Management Information Systems,University of Lisbon,"Lisbon, Portugal",2 years,3900 EUR / year,This Management Information Systems Masters fr...
1,Chemical and Biological Engineering,Universidade Nova de Lisboa,"Lisbon, Portugal",2 years,1250 EUR / year,Students attending the Master of Science in Ch...
2,International Hospitality Management,SHG Universities,Multiple locations,2 years,8900 EUR / year,The International Hospitality Management cours...
3,International Master of Business Administration,Porto Business School,"Senhora da Hora, Portugal",11 months,30 000 EUR / year,This International Master of Business Administ...
4,Building Rehabilitation,Universidade Nova de Lisboa,"Lisbon, Portugal",1½ year,1500 EUR / year,The Master Program in Building Rehabilitation ...
...,...,...,...,...,...,...
83,International and European Law,Universidade Nova de Lisboa - NOVA School of Law,"Lisbon, Portugal",2 years,3100 EUR / year,The Master’s in International and European Law...
84,Accounting,ISEG - Lisbon School of Economics and Management,"Lisbon, Portugal",1½ year,4566 EUR / year,The Accounting MSc programme graduates from IS...
85,Social Law and Innovation,Universidade Nova de Lisboa - NOVA School of Law,"Lisbon, Portugal",2 years,3100 EUR / year,The Social Law and Innovation program from Uni...
86,Law Applied to Technology (Law and Tech),Universidade Nova de Lisboa - NOVA School of Law,"Lisbon, Portugal",2 years,7200 EUR / year,At the Law Applied to Technology (Law and Tech...


In [7]:
df_old = pd.read_csv("first_sixteen_pages.csv")
df_combined = pd.concat([df_old, df], ignore_index=True)
df_combined

,Master Name,University,Location,Duration,Tuition Fee,About
0,Digital Society,Universidade Nova de Lisboa,"Lisbon, Portugal",2 years,1250 EUR / year,The Digital Society programme at Universidade ...
1,Industrial Engineering and Management,Universidade Nova de Lisboa,"Lisbon, Portugal",2 years,1250 EUR / year,The cross-sectional profile of the Master in I...
2,Executive Master of Business Administration,Porto Business School,"Senhora da Hora, Portugal","1 year, 5 months",30 000 EUR / year,This Executive Master of Business Administrati...
3,Strategic Talent Management for the Next Era,Porto Business School,"Senhora da Hora, Portugal",½ year,9500 EUR / year,This Strategic Talent Management for the Next ...
4,Food Engineering,Catholic University of Portugal,"Lisbon, Portugal",2 years,4590 EUR / year,The Food Engineering current cycle of studies ...
...,...,...,...,...,...,...
443,International and European Law,Universidade Nova de Lisboa - NOVA School of Law,"Lisbon, Portugal",2 years,3100 EUR / year,The Master’s in International and European Law...
444,Accounting,ISEG - Lisbon School of Economics and Management,"Lisbon, Portugal",1½ year,4566 EUR / year,The Accounting MSc programme graduates from IS...
445,Social Law and Innovation,Universidade Nova de Lisboa - NOVA School of Law,"Lisbon, Portugal",2 years,3100 EUR / year,The Social Law and Innovation program from Uni...
446,Law Applied to Technology (Law and Tech),Universidade Nova de Lisboa - NOVA School of Law,"Lisbon, Portugal",2 years,7200 EUR / year,At the Law Applied to Technology (Law and Tech...


In [8]:
duplicates = df_combined[df_combined.duplicated(subset=["Master Name", "University"], keep=False)]

display(duplicates)

,Master Name,University,Location,Duration,Tuition Fee,About
30,European Union Law,University of Minho,"Braga, Portugal",2 years,1250 EUR / year,This European Union Law Master degree at Unive...
55,Physical Activity and Sports,University of Madeira,"Funchal, Portugal",2 years,2000 EUR / year,The main aim of this master’s degree in Physic...
56,Global Online Master of Business Administration,Porto Business School,Online,"1 year, 5 months",26 500 EUR / year,This Global Online Master of Business Administ...
57,Cultural Management,University of Madeira,"Funchal, Portugal",2 years,2000 EUR / year,The cycle of studies of the master's degree in...
58,International Hospitality Management,SHG Universities,Multiple locations,2 years,8900 EUR / year,The International Hospitality Management cours...
...,...,...,...,...,...,...
443,International and European Law,Universidade Nova de Lisboa - NOVA School of Law,"Lisbon, Portugal",2 years,3100 EUR / year,The Master’s in International and European Law...
444,Accounting,ISEG - Lisbon School of Economics and Management,"Lisbon, Portugal",1½ year,4566 EUR / year,The Accounting MSc programme graduates from IS...
445,Social Law and Innovation,Universidade Nova de Lisboa - NOVA School of Law,"Lisbon, Portugal",2 years,3100 EUR / year,The Social Law and Innovation program from Uni...
446,Law Applied to Technology (Law and Tech),Universidade Nova de Lisboa - NOVA School of Law,"Lisbon, Portugal",2 years,7200 EUR / year,At the Law Applied to Technology (Law and Tech...


In [9]:
df_clean = df_combined.drop_duplicates(subset=["Master Name", "University"], keep="first")

In [10]:
df_clean

,Master Name,University,Location,Duration,Tuition Fee,About
0,Digital Society,Universidade Nova de Lisboa,"Lisbon, Portugal",2 years,1250 EUR / year,The Digital Society programme at Universidade ...
1,Industrial Engineering and Management,Universidade Nova de Lisboa,"Lisbon, Portugal",2 years,1250 EUR / year,The cross-sectional profile of the Master in I...
2,Executive Master of Business Administration,Porto Business School,"Senhora da Hora, Portugal","1 year, 5 months",30 000 EUR / year,This Executive Master of Business Administrati...
3,Strategic Talent Management for the Next Era,Porto Business School,"Senhora da Hora, Portugal",½ year,9500 EUR / year,This Strategic Talent Management for the Next ...
4,Food Engineering,Catholic University of Portugal,"Lisbon, Portugal",2 years,4590 EUR / year,The Food Engineering current cycle of studies ...
...,...,...,...,...,...,...
355,Nursing,University of Minho,"Braga, Portugal",2 years,1250 EUR / year,The Master course in Nursing at University of ...
356,Building Rehabilitation,Universidade Nova de Lisboa,"Lisbon, Portugal",1½ year,1500 EUR / year,The Master Program in Building Rehabilitation ...
357,Civil Engineering,Universidade Nova de Lisboa,"Lisbon, Portugal",2 years,1250 EUR / year,The Master in Civil Engineering from Universid...
358,Chemical and Biological Engineering,Universidade Nova de Lisboa,"Lisbon, Portugal",2 years,1250 EUR / year,Students attending the Master of Science in Ch...


In [11]:
df_clean.to_csv("master_portal.csv", index=False)

In [12]:
master_portal = pd.read_csv("master_portal.csv")
EduPortugal = pd.read_csv("EduPortugal_masters.csv")


In [13]:
display(master_portal)
display(EduPortugal)

,Master Name,University,Location,Duration,Tuition Fee,About
0,Digital Society,Universidade Nova de Lisboa,"Lisbon, Portugal",2 years,1250 EUR / year,The Digital Society programme at Universidade ...
1,Industrial Engineering and Management,Universidade Nova de Lisboa,"Lisbon, Portugal",2 years,1250 EUR / year,The cross-sectional profile of the Master in I...
2,Executive Master of Business Administration,Porto Business School,"Senhora da Hora, Portugal","1 year, 5 months",30 000 EUR / year,This Executive Master of Business Administrati...
3,Strategic Talent Management for the Next Era,Porto Business School,"Senhora da Hora, Portugal",½ year,9500 EUR / year,This Strategic Talent Management for the Next ...
4,Food Engineering,Catholic University of Portugal,"Lisbon, Portugal",2 years,4590 EUR / year,The Food Engineering current cycle of studies ...
...,...,...,...,...,...,...
355,Nursing,University of Minho,"Braga, Portugal",2 years,1250 EUR / year,The Master course in Nursing at University of ...
356,Building Rehabilitation,Universidade Nova de Lisboa,"Lisbon, Portugal",1½ year,1500 EUR / year,The Master Program in Building Rehabilitation ...
357,Civil Engineering,Universidade Nova de Lisboa,"Lisbon, Portugal",2 years,1250 EUR / year,The Master in Civil Engineering from Universid...
358,Chemical and Biological Engineering,Universidade Nova de Lisboa,"Lisbon, Portugal",2 years,1250 EUR / year,Students attending the Master of Science in Ch...


,master,university,about
0,Ciências Empresariais,ISEG – Lisbon School of Economics and Management.,Apresentação do Curso\nO Mestrado em Ciências ...
1,"Contabilidade, Fiscalidade e Finanças Empresar...",ISEG – Lisbon School of Economics and Management.,Apresentação do Curso\nA contabilidade moderna...
2,Desenvolvimento e Cooperação Internacional,ISEG – Lisbon School of Economics and Management.,Apresentação do Curso\nSe queres deixar a tua ...
3,Economia e Políticas Públicas,ISEG – Lisbon School of Economics and Management.,Apresentação do Curso\nSe queres aprofundar o ...
4,Economia Internacional e Estudos Europeus,ISEG – Lisbon School of Economics and Management.,Apresentação do Curso\nSe estás interessado(a)...
...,...,...,...
1454,Didática do Inglês,NOVA FCSH – Faculdade de Ciências Sociais e Hu...,Apresentação do Curso\nAprofundar conhecimento...
1455,Ordenamento do Território e Sistemas de Inform...,NOVA FCSH – Faculdade de Ciências Sociais e Hu...,Apresentação do Curso\nO curso funciona em reg...
1456,Gestão de Marketing (E-Learning),IPAM – Instituto Português de Administração de...,Apresentação do Curso\nO Mestrado em Gestão de...
1457,Português Língua Não Materna,Universidade Aberta,Objetivos\nO Curso é desenvolvido a partir do ...


In [ ]:
master_portal = master_portal.rename(columns={
    "Master Name": "master",
    "University": "university",
    "About": "about"
})


,master,university,Location,Duration,Tuition Fee,about
0,Digital Society,Universidade Nova de Lisboa,"Lisbon, Portugal",2 years,1250 EUR / year,The Digital Society programme at Universidade ...
1,Industrial Engineering and Management,Universidade Nova de Lisboa,"Lisbon, Portugal",2 years,1250 EUR / year,The cross-sectional profile of the Master in I...
2,Executive Master of Business Administration,Porto Business School,"Senhora da Hora, Portugal","1 year, 5 months",30 000 EUR / year,This Executive Master of Business Administrati...
3,Strategic Talent Management for the Next Era,Porto Business School,"Senhora da Hora, Portugal",½ year,9500 EUR / year,This Strategic Talent Management for the Next ...
4,Food Engineering,Catholic University of Portugal,"Lisbon, Portugal",2 years,4590 EUR / year,The Food Engineering current cycle of studies ...
...,...,...,...,...,...,...
355,Nursing,University of Minho,"Braga, Portugal",2 years,1250 EUR / year,The Master course in Nursing at University of ...
356,Building Rehabilitation,Universidade Nova de Lisboa,"Lisbon, Portugal",1½ year,1500 EUR / year,The Master Program in Building Rehabilitation ...
357,Civil Engineering,Universidade Nova de Lisboa,"Lisbon, Portugal",2 years,1250 EUR / year,The Master in Civil Engineering from Universid...
358,Chemical and Biological Engineering,Universidade Nova de Lisboa,"Lisbon, Portugal",2 years,1250 EUR / year,Students attending the Master of Science in Ch...


In [17]:
df_final = pd.concat([master_portal, EduPortugal], ignore_index=True)
df_final

,master,university,Location,Duration,Tuition Fee,about
0,Digital Society,Universidade Nova de Lisboa,"Lisbon, Portugal",2 years,1250 EUR / year,The Digital Society programme at Universidade ...
1,Industrial Engineering and Management,Universidade Nova de Lisboa,"Lisbon, Portugal",2 years,1250 EUR / year,The cross-sectional profile of the Master in I...
2,Executive Master of Business Administration,Porto Business School,"Senhora da Hora, Portugal","1 year, 5 months",30 000 EUR / year,This Executive Master of Business Administrati...
3,Strategic Talent Management for the Next Era,Porto Business School,"Senhora da Hora, Portugal",½ year,9500 EUR / year,This Strategic Talent Management for the Next ...
4,Food Engineering,Catholic University of Portugal,"Lisbon, Portugal",2 years,4590 EUR / year,The Food Engineering current cycle of studies ...
...,...,...,...,...,...,...
1814,Didática do Inglês,NOVA FCSH – Faculdade de Ciências Sociais e Hu...,NaN,NaN,NaN,Apresentação do Curso\nAprofundar conhecimento...
1815,Ordenamento do Território e Sistemas de Inform...,NOVA FCSH – Faculdade de Ciências Sociais e Hu...,NaN,NaN,NaN,Apresentação do Curso\nO curso funciona em reg...
1816,Gestão de Marketing (E-Learning),IPAM – Instituto Português de Administração de...,NaN,NaN,NaN,Apresentação do Curso\nO Mestrado em Gestão de...
1817,Português Língua Não Materna,Universidade Aberta,NaN,NaN,NaN,Objetivos\nO Curso é desenvolvido a partir do ...


In [18]:
df_final.to_csv('masters_portugal.csv', index=False)